In [1]:
import sys
import numpy as np
import pandas as pd
import time
sys.path.append('../src')
import KGBN

# Define PBN rules with initial probabilities
rules = """
N1 = N1, 1
N2 = N2, 1
N3 = N1, 0.5
N3 = N1 & !N2, 0.5
"""

# Load PBN
pbn = KGBN.load_network(rules)

No initial state provided, using a random initial state
PBN loaded successfully. There are 3 genes in the network.


In [2]:
# synthetic pbn data
rules_syn = """
N1 = N1, 1
N2 = N2, 1
N3 = N1, 0.1
N3 = N1 & !N2, 0.9
"""
# Load PBN
pbn_syn = KGBN.load_network(rules_syn)

No initial state provided, using a random initial state
PBN loaded successfully. There are 3 genes in the network.


In [3]:
# Exp1
calc = KGBN.SteadyStateCalculator(pbn_syn)
exp1 = calc.compute_steady_state(method='monte_carlo',n_runs=5,n_steps=1000,p_noise=0.05,seed=99)
exp1.sum()

1.1429141716566866

In [4]:
# Exp2
calc = KGBN.SteadyStateCalculator(pbn_syn)
calc.set_experimental_conditions(stimuli=['N1','N2'])
exp2 = calc.compute_steady_state(method='monte_carlo',n_runs=5,n_steps=1000,p_noise=0.05,seed=99)
exp2.sum()

2.1417165668662674

In [5]:
# Exp3
calc = KGBN.SteadyStateCalculator(pbn_syn)
calc.set_experimental_conditions(stimuli=['N1'],inhibitors=['N2'])
exp3 = calc.compute_steady_state(method='monte_carlo',n_runs=5,n_steps=1000,p_noise=0.05,seed=99)
exp3.sum()

1.9477045908183634

In [9]:
# Example 1: Parse formula directly from CSV with use_formula=True
# The CSV column Measured_nodes should contain the formula string, and Measured_values must have a single value per row

# Create experimental data
experiments = pd.DataFrame({
    'Experiments': [1, 2, 3],
    'Stimuli': ['', 'N1,N2', 'N1'],
    'Inhibitors': ['', '', 'N2'],
    'Measured_nodes': ['N1 + N2 + N3', 'N1 + N2 + N3', 'N1 + N2 + N3'],
    'Measured_values': ['1.14', '2.14', '1.95']
})
experiments.to_csv('files/experiments_formula.csv', index=False)

# load experiments
from KGBN.experiment_data import ExperimentData
experiments_formula = ExperimentData.load_from_csv('files/experiments_formula.csv', use_formula=True)

In [10]:
# Configure optimizer
config = {
    'seed': 99,
    'max_try': 1,  # Maximum number of attempts if optimization fails
    'early_stopping': True,  # Control early stopping for both DE and PSO
    'success_threshold': 2e-4,
    'de_params': {
        'strategy': 'best1bin',
        'maxiter': 50,
        'popsize': 20,
        'tol': 0.00001,
        'mutation': (0.5, 1),
        'recombination': 0.7,
        'init': 'sobol',
        'updating': 'deferred',
        'workers': -1, # Use all available cores for parallelization
        'polish': False,  # Disable polish step for faster runs
    },
    'steady_state': {
        'method': 'monte_carlo',
        'monte_carlo_params': {
            'n_runs': 5, 
            'n_steps': 1000,
            'p_noise': 0.05,
            'seed': 99
        },
    },
}

# optimize
optimizer_formula = KGBN.ParameterOptimizer(
    pbn,
    experiments_formula,
    config=config,
    nodes_to_optimize=['N3'],
    verbose=False
)
result_formula = optimizer_formula.optimize(method='differential_evolution')


Optimization attempt 1/1
Running DE optimization...


  Predicted formula value: 0.052295409181636734, Measured value: 1.14
  Predicted formula value: 0.052295409181636734, Measured value: 1.14
  Predicted formula value: 0.052295409181636734, Measured value: 1.14
  Predicted formula value: 0.052295409181636734, Measured value: 1.14
  Predicted formula value: 0.052295409181636734, Measured value: 1.14
  Predicted formula value: 0.052295409181636734, Measured value: 1.14
  Predicted formula value: 0.052295409181636734, Measured value: 1.14
  Predicted formula value: 0.052295409181636734, Measured value: 1.14
  Predicted formula value: 0.052295409181636734, Measured value: 1.14
  Predicted formula value: 0.052295409181636734, Measured value: 1.14
  Predicted formula value: 2.788822355289421, Measured value: 2.14
  Predicted formula value: 2.7157684630738523, Measured value: 2.14
  Predicted formula value: 0.052295409181636734, Measured value: 1.14
  Predicted formula value: 2.59680638722

In [6]:
# Example 2: Use a global formula override
# Optimizes against the score N1 + N2 - N3 using the CSV's Measured_values (one value per row)
optimizer_formula = KGBN.ParameterOptimizer(
    pbn,
    'files/experiments_formula.csv',
    config=config,
    nodes_to_optimize=['N3'],
    verbose=False,
    Measured_formula='N1 + N2 + N3'
)
result_formula = optimizer_formula.optimize(method='differential_evolution')


Optimization attempt 1/1
Running DE optimization...




Process SpawnPoolWorker-37:
Process SpawnPoolWorker-41:
Process SpawnPoolWorker-29:
Process SpawnPoolWorker-34:
Process SpawnPoolWorker-36:
Process SpawnPoolWorker-31:
Process SpawnPoolWorker-39:
Process SpawnPoolWorker-33:
Process SpawnPoolWorker-30:
Process SpawnPoolWorker-38:
Process SpawnPoolWorker-35:
Process SpawnPoolWorker-40:
Process SpawnPoolWorker-32:
Process SpawnPoolWorker-42:
Traceback (most recent call last):
Traceback (most recent call last):
  File "/opt/anaconda3/envs/logicmodelmerger/lib/python3.13/multiprocessing/process.py", line 313, in _bootstrap
    self.run()
    ~~~~~~~~^^
  File "/opt/anaconda3/envs/logicmodelmerger/lib/python3.13/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/logicmodelmerger/lib/python3.13/multiprocessing/pool.py", line 114, in worker
    task = get()
  File "/opt/anaconda3/envs/logicmodelmerger/lib/python3.13/multiprocessin

KeyboardInterrupt: 